# Tutorial 04 — Scanpy Spatial Basic Analysis

**Source:** https://scanpy-tutorials.readthedocs.io/en/latest/spatial/basic-analysis.html  
**Author:** Giovanni Palla  
**Datasets:** 10x Visium Mouse Brain + MERFISH Mouse Cortex

---

## What this notebook covers
1. Read and inspect Visium spatial transcriptomics data
2. QC and preprocessing
3. Dimensionality reduction and Leiden clustering
4. Visualization in spatial coordinates with `sc.pl.spatial()`
5. Cluster marker gene identification
6. Spatial gene expression mapping
7. MERFISH example: mouse cortex spatial analysis

## 0. Imports

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import squidpy as sq
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sc.logging.print_versions()
sc.set_figure_params(facecolor='white', figsize=(8, 8))
sc.settings.verbosity = 3
sc.settings.figdir = '../results/figures/04_scanpy_spatial/'

## Part A — Visium Mouse Brain

### 1. Read the Data

In [ ]:
# Load pre-processed Visium mouse brain dataset via Squidpy
adata = sq.datasets.visium_hne_adata()

print(adata)
print(f"\nSpatial coordinates shape: {adata.obsm['spatial'].shape}")

In [ ]:
# Basic data inspection
print("Observation metadata:")
print(adata.obs.head())
print("\nFirst 5 gene names:")
print(adata.var_names[:5].tolist())

### 2. QC and Preprocessing

In [ ]:
# Calculate QC metrics
adata.var['mt'] = adata.var_names.str.startswith('Mt-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

# Visualize QC
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
sns.histplot(adata.obs['total_counts'], bins=100, ax=axes[0])
axes[0].set_title('Total counts')
sns.histplot(adata.obs['n_genes_by_counts'], bins=100, ax=axes[1])
axes[1].set_title('Genes per spot')
sns.histplot(adata.obs['total_counts'][adata.obs['total_counts'] < 10000], bins=100, ax=axes[2])
axes[2].set_title('Total counts (< 10k)')
sns.histplot(adata.obs['pct_counts_mt'], bins=100, ax=axes[3])
axes[3].set_title('MT%')
plt.tight_layout()
plt.savefig('../results/figures/04_scanpy_spatial/qc_histograms.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Filter and preprocess
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added='leiden_clusters', resolution=0.5)

print(f"Leiden clusters: {adata.obs['leiden_clusters'].nunique()}")

### 3. Visualization in Spatial Coordinates

In [ ]:
# UMAP coloured by cluster
sc.pl.umap(
    adata,
    color=['leiden_clusters', 'cluster'],
    save='_visium_umap.png'
)

In [ ]:
# Spatial scatter — clusters on tissue
sq.pl.spatial_scatter(
    adata,
    color='cluster',
    save='../results/figures/04_scanpy_spatial/spatial_clusters.png'
)

In [ ]:
# QC metrics on tissue
sq.pl.spatial_scatter(
    adata,
    color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
    save='../results/figures/04_scanpy_spatial/spatial_qc.png'
)

### 4. Cluster Marker Genes

In [ ]:
sc.tl.rank_genes_groups(adata, 'cluster', method='wilcoxon')
sc.pl.rank_genes_groups(
    adata,
    n_genes=5,
    sharey=False,
    save='_marker_genes.png'
)

In [ ]:
# Spatial expression of key marker genes
marker_genes = ['Nrgn', 'Hpca', 'Enpp2', 'Mbp', 'Ttr']

# Filter to genes present in data
marker_genes = [g for g in marker_genes if g in adata.var_names]

sq.pl.spatial_scatter(
    adata,
    color=marker_genes,
    save='../results/figures/04_scanpy_spatial/spatial_marker_genes.png'
)

---

## Part B — MERFISH Mouse Cortex

MERFISH (Multiplexed Error-Robust Fluorescence In Situ Hybridization) is a single-cell spatial transcriptomics technology with subcellular resolution. Here we demonstrate a basic analysis of mouse cortex data.

In [ ]:
# Load MERFISH mouse cortex dataset
adata_merfish = sq.datasets.merfish()

print(adata_merfish)
print(f"\nCell types: {adata_merfish.obs['Cell_class'].unique().tolist()}")

In [ ]:
# Visualize cell type annotation spatially
sq.pl.spatial_scatter(
    adata_merfish,
    color='Cell_class',
    shape=None,
    size=2,
    save='../results/figures/04_scanpy_spatial/merfish_cell_types.png'
)

In [ ]:
# Spatial gene expression — example genes
genes_of_interest = adata_merfish.var_names[:4].tolist()

sq.pl.spatial_scatter(
    adata_merfish,
    color=genes_of_interest,
    shape=None,
    size=2,
    save='../results/figures/04_scanpy_spatial/merfish_gene_expression.png'
)

In [ ]:
# Spatial neighbor graph and neighborhood enrichment
sq.gr.spatial_neighbors(adata_merfish, coord_type='generic', delaunay=True)
sq.gr.nhood_enrichment(adata_merfish, cluster_key='Cell_class')
sq.pl.nhood_enrichment(
    adata_merfish,
    cluster_key='Cell_class',
    method='average',
    save='../results/figures/04_scanpy_spatial/merfish_nhood_enrichment.png'
)

## Save Results

In [ ]:
adata.write('../results/scanpy_spatial_visium.h5ad')
adata_merfish.write('../results/scanpy_spatial_merfish.h5ad')
print('Results saved.')